In [9]:
import pandas as pd
import numpy as np

In [11]:


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv(r"C:\Users\hp\Desktop\stage etis\open_dataset\goodscents_jadbio_ready.csv", sep=";")

# ============================================================
# Define columns
# ============================================================
label_cols = [
    "floral",
    "fruity",
    "sweet",
    "woody",
    "green",
    "spicy",
    "animal_musk",
    "earthy",
    "citrus",
    "chemical",
    "gourmand",
    "powdery_amber"
]

meta_cols = ["SMILES"]

feature_cols = [c for c in df.columns if c not in label_cols + meta_cols]

missing_feature_cols = [
    col for col in feature_cols
    if df[col].isnull().any()
]

print(f"Feature columns with missing values: {len(missing_feature_cols)}")

# drop features with missing values (ignore any that might not exist)
df = df.drop(columns=missing_feature_cols, errors='ignore')

# recompute feature columns after drop
feature_cols = [c for c in df.columns if c not in label_cols + meta_cols]

# recompute missing_feature_cols and report
missing_feature_cols = [col for col in feature_cols if df[col].isnull().any()]
print(f"Feature columns with missing values: {len(missing_feature_cols)}")

X = df[feature_cols]
Y = df[label_cols]

# ============================================================
# Basic dataset info
# ============================================================
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

print(f"Total samples          : {len(df):,}")
print(f"Total columns          : {df.shape[1]:,}")
print(f"Number of features     : {len(feature_cols):,}")
print(f"Number of labels       : {len(label_cols)}")



Feature columns with missing values: 327
Feature columns with missing values: 0
DATASET OVERVIEW
Total samples          : 4,981
Total columns          : 691
Number of features     : 678
Number of labels       : 12


In [12]:
# ============================================================
# Missing values
# ============================================================
print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

total_missing = df.isna().sum().sum()
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)

print(f"Total missing values   : {total_missing:,}")
print(f"Columns with missing   : {len(missing_cols)}")

if len(missing_cols) > 0:
    print("\nTop missing columns:")
    print(missing_cols.head(20))




MISSING VALUES
Total missing values   : 0
Columns with missing   : 0


In [13]:
# ============================================================
# Infinite values
# ============================================================
print("\n" + "=" * 80)
print("INFINITE VALUES")
print("=" * 80)

numeric_df = df.select_dtypes(include=np.number)

total_inf = np.isinf(numeric_df).sum().sum()

print(f"Total infinite values  : {total_inf:,}")

if total_inf > 0:
    inf_cols = numeric_df.columns[np.isinf(numeric_df).any()]
    print("\nColumns containing infinities:")
    print(list(inf_cols[:20]))




INFINITE VALUES
Total infinite values  : 0


In [14]:
# ============================================================
# Constant features
# ============================================================
print("\n" + "=" * 80)
print("CONSTANT FEATURES")
print("=" * 80)

constant_features = [
    col for col in feature_cols
    if X[col].nunique(dropna=False) <= 1
]

print(f"Constant features      : {len(constant_features)}")

if len(constant_features) > 0:
    print("\nExamples:")
    print(constant_features[:20])




CONSTANT FEATURES
Constant features      : 6

Examples:
['MACCS_001', 'MACCS_002', 'MACCS_004', 'MACCS_006', 'MACCS_021', 'MACCS_068']


In [15]:
# ============================================================
# Duplicate rows
# ============================================================
print("\n" + "=" * 80)
print("DUPLICATE ROWS")
print("=" * 80)

duplicates = df.duplicated().sum()
print(f"Duplicate rows         : {duplicates}")

# ============================================================
# Feature statistics
# ============================================================
print("\n" + "=" * 80)
print("FEATURE DISTRIBUTION")
print("=" * 80)

feature_min = X.min(numeric_only=True).min()
feature_max = X.max(numeric_only=True).max()

print(f"Global feature min     : {feature_min:.6f}")
print(f"Global feature max     : {feature_max:.6f}")

# ============================================================
# Label statistics
# ============================================================
print("\n" + "=" * 80)
print("LABEL DISTRIBUTION")
print("=" * 80)

label_counts = Y.sum().sort_values(ascending=False)

label_stats = pd.DataFrame({
    "Positive Samples": label_counts,
    "Prevalence (%)": (100 * label_counts / len(Y)).round(2)
})

print(label_stats)

# ============================================================
# Multilabel statistics
# ============================================================
print("\n" + "=" * 80)
print("MULTILABEL CHARACTERISTICS")
print("=" * 80)

labels_per_sample = Y.sum(axis=1)

label_cardinality = labels_per_sample.mean()
label_density = label_cardinality / len(label_cols)

print(f"Label cardinality      : {label_cardinality:.3f}")
print(f"Label density          : {label_density:.3f}")

print("\nLabels per molecule:")
print(labels_per_sample.value_counts().sort_index())

# ============================================================
# Class imbalance ratio
# ============================================================
print("\n" + "=" * 80)
print("CLASS IMBALANCE")
print("=" * 80)

max_count = label_counts.max()

imbalance_df = pd.DataFrame({
    "Positive Samples": label_counts,
    "Imbalance Ratio": (max_count / label_counts).round(2)
})

print(imbalance_df.sort_values("Imbalance Ratio", ascending=False))

# ============================================================
# Fingerprint vs Mordred dimensions
# ============================================================
print("\n" + "=" * 80)
print("FEATURE BREAKDOWN")
print("=" * 80)

maccs_cols = [c for c in feature_cols if c.startswith("MACCS")]
morgan_cols = [c for c in feature_cols if c.startswith("morgan")]

mordred_cols = [
    c for c in feature_cols
    if c not in maccs_cols and c not in morgan_cols
]

print(f"MACCS features         : {len(maccs_cols)}")
print(f"Morgan features        : {len(morgan_cols)}")
print(f"Mordred descriptors    : {len(mordred_cols)}")

print("\nExploration complete.")


DUPLICATE ROWS
Duplicate rows         : 0

FEATURE DISTRIBUTION
Global feature min     : 0.000000
Global feature max     : 46.000000

LABEL DISTRIBUTION
               Positive Samples  Prevalence (%)
fruity                     2282           45.81
green                      2252           45.21
chemical                   2145           43.06
sweet                      1928           38.71
floral                     1304           26.18
gourmand                   1206           24.21
spicy                      1075           21.58
woody                      1055           21.18
powdery_amber              1026           20.60
earthy                     1005           20.18
animal_musk                 772           15.50
citrus                      513           10.30

MULTILABEL CHARACTERISTICS
Label cardinality      : 3.325
Label density          : 0.277

Labels per molecule:
1      779
2      920
3     1099
4     1038
5      625
6      331
7      138
8       44
9        5
10       1
